# AAPL Data Exploration

This notebook is the consolidated research log of everything we learned
about the Apple daily price series before we started building the
modeling pipeline. The original notebooks (`apple_week1`, `apple_week2`,
`apple_week3`, `apple_find_duplicates`) all worked on the same questions
from slightly different angles, so this file pulls them together.

Sections:
1. Loading the raw file and basic shape checks
2. Completeness: duplicates, missing trading days, forward fill
3. Tuesday to Thursday weekly construction
4. Week type distribution (good, bad, flat)
5. Streak analysis and geometric fit
6. Streak transition tables
7. Year level classification
8. First model attempt with Random Forest
9. What we took forward into the pipeline

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import geom
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, roc_curve, auc,
    precision_score, recall_score, f1_score,
)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

## 1. Loading the raw file

The original CSV is the daily AAPL series from 1984-09-07 through
2022-04-22. The columns come wrapped in angle brackets, so we strip
those off. The `DATE` column is read as an integer like `19840907`
and we parse it into a real datetime.

In [ ]:
apple = pd.read_csv(r"D:\data\aapl.us.txt")
apple['<DATE>'] = pd.to_datetime(apple['<DATE>'], format="%Y%m%d")
apple.columns = apple.columns.str.strip().str.strip("<>").str.upper()
apple['DayOfWeek'] = apple['DATE'].dt.day_name()
apple.head()

In [ ]:
apple.shape, apple.isnull().values.any()

Shape: 9484 rows. No NaNs in the raw file. Whatever gaps exist will
show up as missing dates rather than missing values.

## 2. Completeness check

We expect five rows per business week. The first thing we did was
build a calendar of every weekday between the start and end of the
series and join the trading data onto it. Whatever does not match
is either a weekend (already filtered) or a US market holiday.

In [ ]:
start = "1984-09-07"
end = "2022-04-22"
dates = pd.date_range(start=start, end=end, freq="D")
calendar = pd.DataFrame({
    'DATE': dates,
    'weekday': dates.strftime('%A'),
})
merged = pd.merge(calendar, apple[['DATE','OPEN','CLOSE','VOL','HIGH','LOW']], on='DATE', how='left')
merged = merged[~merged['weekday'].isin(['Saturday','Sunday'])].reset_index(drop=True)
print('rows with NaNs:', int(merged.isnull().any(axis=1).sum()))
merged.isnull().sum()

About 332 rows out of roughly 9800 weekday rows are missing, which is
around 3.4 percent. Those are the holidays. The interesting part is
the weekday distribution of the holes:

In [ ]:
null_rows = merged[merged.isnull().any(axis=1)]
weekday_holes = null_rows['weekday'].value_counts()
weekday_holes

Most holes fall on Monday, Tuesday, Wednesday and Friday, with
Thursday hit by Thanksgiving every year. That matters because our
trade is buy Tuesday open and sell Thursday open, so a missing
Tuesday or Thursday means a week we cannot trade.

We checked whether the holidays cluster (consecutive nulls). They do,
around the 9/11 closure (Sep 12 to 14, 2001) and a couple of weather
events. For the rest the holes are isolated. To keep the time series
shape we forward fill the holiday rows; this preserves the previous
trading day's quote which is what most analyses end up doing anyway.

In [ ]:
merged_impute = merged.fillna(method='ffill')
merged_impute.isnull().sum().sum()

After forward fill there are no remaining NaNs. We saved the result
as `cleaned_apple_high_low.csv` and that file is what the pipeline
loads in production.

## 3. Tuesday to Thursday weekly construction

The trade is: open a position on Tuesday's open, close on Thursday's
open. For every calendar week we build a row with the Tuesday open
and the Thursday open of that same week, then compute
`thu_open / tue_open` as the multiplier and `(multiplier - 1) * 100`
as the net percent return.

In [ ]:
apple['weekday'] = apple['DATE'].dt.weekday
apple['week'] = apple['DATE'].dt.to_period('W-MON')
tue_open = (apple.loc[apple['weekday']==1].groupby('week')['OPEN'].first().rename('buy_tue_open'))
thu_open = (apple.loc[apple['weekday']==3].groupby('week')['OPEN'].first().rename('sell_thu_open'))
weekly = pd.concat([tue_open, thu_open], axis=1).dropna()
weekly['net%'] = ((weekly['sell_thu_open']/weekly['buy_tue_open']) - 1.0) * 100.0
weekly.head()

## 4. Week type distribution

We label every week as good, bad or flat depending on the sign of
`net%`. The flat bucket only contains zero return weeks. There are
45 of them in the whole series; mostly very early years where the
stock barely moved between Tuesday and Thursday open.

In [ ]:
weekly['week_type'] = np.where(
    weekly['net%'] > 0, 'good week',
    np.where(weekly['net%'] < 0, 'bad week', 'flat week')
)
counts = weekly['week_type'].value_counts()
counts

Roughly 51 percent good weeks, 46 percent bad, 2 percent flat. Almost
balanced, with a small tilt toward profitable weeks. That tilt is what
we are trying to amplify with a model.

For modeling we collapse the flat bucket into bad (treat zero return as
no profit) and use a binary `week_type_bool` target. That is what the
pipeline does today.

## 5. Streak analysis and geometric fit

Run length encoding of the good and bad sequence gives us per streak
lengths. The streaks are mostly short (around 1 to 3 weeks), with one
outlier of 11 consecutive bad weeks starting 1990-08-14.

In [ ]:
runs = (weekly['week_type'] != weekly['week_type'].shift()).cumsum()
streaks = weekly.groupby([runs]).agg(
    week_type=('week_type','first'),
    length=('week_type','size')
).reset_index(drop=True)
streaks['length'].describe()

In [ ]:
longest = streaks.sort_values('length', ascending=False).head(5)
longest

We tried to fit a geometric distribution to the combined streak
lengths to see whether week to week outcomes look memoryless. Solving
for the maximum likelihood `p` lands around 0.47, which is close
to the empirical good week rate. In other words the streaks are not
obviously richer than a coin with the same long run rate.

In [ ]:
observed_counts = streaks.groupby('length').size()
observed_probs = observed_counts / observed_counts.sum()
lengths = observed_counts.index.values

def neg_log_likelihood(p):
    if p <= 0 or p >= 1:
        return np.inf
    expected = geom.pmf(lengths, p)
    expected = expected / expected.sum()
    return -np.sum(observed_probs * np.log(expected))

result = minimize(neg_log_likelihood, x0=0.5, bounds=[(0.01, 0.99)])
best_p = float(result.x[0])
print('estimated p:', best_p)

## 6. Streak transition tables

After a good streak of length `n`, what does the next bad streak
tend to look like? Building a contingency table:

In [ ]:
streaks['next_bad_length'] = streaks['length'].shift(-1).where(streaks['week_type']=='good week')
streaks['next_good_length'] = streaks['length'].shift(-1).where(streaks['week_type']=='bad week')
good_to_bad = streaks[streaks['week_type']=='good week'].dropna(subset=['next_bad_length'])
bad_to_good = streaks[streaks['week_type']=='bad week'].dropna(subset=['next_good_length'])
print('corr good length vs next bad length:', good_to_bad['length'].corr(good_to_bad['next_bad_length']))
print('corr bad length vs next good length:', bad_to_good['length'].corr(bad_to_good['next_good_length']))

Both correlations are within 0.01 of zero, which is consistent with
the geometric story: streak length contains essentially no
information about what comes next. This was the strongest signal we
got from the early exploration that simple chronological features
alone are not enough; that is why the support resistance envelope
features were introduced later.

## 7. Year level classification

We rolled the weekly labels up into year long windows: a year is
good if it has more good weeks than bad weeks in the next 52 week
period. About 60 percent of years come out good. Useful as context
but not a feature.

In [ ]:
weekly_reset = weekly.reset_index()
weekly_reset['week_start'] = weekly_reset['week'].dt.start_time
weekly_reset['good_flag'] = (weekly_reset['week_type']=='good week').astype(int)
weekly_reset['bad_flag']  = (weekly_reset['week_type']=='bad week').astype(int)
labels = []
for i in range(len(weekly_reset)):
    start_date = weekly_reset.loc[i, 'week_start']
    end_date = start_date + pd.Timedelta(weeks=52)
    mask = (weekly_reset['week_start'] >= start_date) & (weekly_reset['week_start'] < end_date)
    g = weekly_reset.loc[mask, 'good_flag'].sum()
    b = weekly_reset.loc[mask, 'bad_flag'].sum()
    if g > b: labels.append('good year')
    elif b > g: labels.append('bad year')
    else: labels.append('flat year')
weekly_reset['year_type'] = labels
weekly_reset['year_type'].value_counts()

## 8. First model attempt: Random Forest

Before settling on the rolling decision tree, we tried a single shot
Random Forest with three simple features (Tuesday open, Monday volume,
previous Friday open). It did not beat random. The cross validated
accuracy hovered around 50 percent and the ROC curve sat on the
diagonal. Recording it here so we remember what did not work.

In [ ]:
mon_vol = (apple.loc[apple['weekday']==0].set_index('week')['VOL'].rename('Mon_Volume'))
fri_open = (apple.loc[apple['weekday']==4].set_index('week')['OPEN'].rename('PrevFri_Open').shift(1))
weekly2 = weekly.copy()
weekly2['week_type_bool'] = (weekly2['week_type']=='good week').astype(int)
weekly_full = (weekly2.join(mon_vol, how='left').join(fri_open, how='left').dropna())
X = weekly_full[['buy_tue_open','Mon_Volume','PrevFri_Open']]
y = weekly_full['week_type_bool']
split_idx = int(len(X)*0.7)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
tscv = TimeSeriesSplit(n_splits=5)
clf = RandomForestClassifier(n_estimators=300, random_state=42)
scores = cross_val_score(clf, X_train, y_train, cv=tscv, scoring='accuracy')
print('CV scores:', scores)
print('mean:', scores.mean())

In [ ]:
clf.fit(X_train, y_train)
y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
print('test ROC AUC:', roc_auc)

AUC near 0.5 confirmed that those three raw features carry no signal.
The model could only chase recall by setting the threshold so low it
predicted almost everything as positive. That convinced us the next
step was either better features (the support resistance envelope work)
or a different formulation (rolling window with a custom score).

## 9. What we took forward

1. The Tuesday to Thursday weekly aggregation is the right unit of
   analysis and we kept the binary good versus not good label.
2. The 332 holiday gaps need forward fill before any feature work.
3. Streak structure on its own contains no usable information, so
   any model has to rely on price level features.
4. A flat train then test split with simple features did not work.
   The pipeline now uses a rolling expanding window with a custom
   precision weighted score function.
5. The two feature families that made it into the pipeline live in
   `aapl_pipeline.feature_calculator`: `TueThuNormalizedFeatures`
   and `SupportResistanceFeatures`.